In [0]:
%run ./_config

In [0]:
# simple pipeline health report
health = spark.sql(f"""
    SELECT
        (SELECT COUNT(*) FROM {SILVER_SCHEMA}.fct_sales) AS fact_rows,
        (SELECT COUNT(DISTINCT store_id)  FROM {SILVER_SCHEMA}.dim_stores  WHERE is_current = TRUE) AS active_stores,
        (SELECT COUNT(DISTINCT product_id) FROM {SILVER_SCHEMA}.dim_products WHERE is_current = TRUE) AS active_products,
        (SELECT COUNT(DISTINCT vendor_id) FROM {SILVER_SCHEMA}.dim_vendors  WHERE is_current = TRUE) AS active_vendors,
        (SELECT SUM(CASE WHEN store_id IS NULL THEN 1 ELSE 0 END) FROM {SILVER_SCHEMA}.fct_sales) AS null_store_fks,
        (SELECT MIN(sale_date) FROM {SILVER_SCHEMA}.fct_sales) AS earliest_sale,
        (SELECT MAX(sale_date) FROM {SILVER_SCHEMA}.fct_sales) AS latest_sale
""")
health.display()

In [0]:
# read upstream task values (works only within Workflow)
try:
    rows_ingested = dbutils.jobs.taskValues.get(
        taskKey="ingest_api",
        key="rows_ingested",
        default=0
    )
    load_date = dbutils.jobs.taskValues.get(
        taskKey="ingest_api",
        key="load_date",
        default="unknown"
    )
    print(f"Ingestion reported: {rows_ingested:,} rows, load date: {load_date}")
except Exception:
    print("Task values unavailable")

In [0]:
# gold layer validation
gold_tables = [
    "gold_monthly_sales", 
    "gold_dow_patterns", 
    "gold_county_performance",
    "gold_city_rankings", 
    "gold_vendor_scorecard",
    "gold_category_market_share", 
    "gold_store_rankings",
    "daily_load_audit"
]

for tbl in gold_tables:
    count = spark.sql(f"SELECT COUNT(*) AS cnt FROM {GOLD_SCHEMA}.{tbl}").collect()[0]["cnt"]
    status = "OK" if count > 0 else "Empty, check pipeline"
    print(f"  {tbl}: {count:,} rows — {status}")

In [ ]:
# cross-layer dq check
spark.sql(f"""
    SELECT
        (SELECT COUNT(*) FROM {SILVER_SCHEMA}.cleaned_sales) AS cleaned_rows,
        (SELECT COUNT(*) FROM {SILVER_SCHEMA}.fct_sales) AS fact_rows,
        (SELECT COUNT(*) FROM {SILVER_SCHEMA}.cleaned_sales) -
            (SELECT COUNT(*) FROM {SILVER_SCHEMA}.fct_sales) AS dropped_by_merge,
        ROUND(
            (SELECT SUM(sale_dollars) FROM {SILVER_SCHEMA}.fct_sales) -
            COALESCE((SELECT SUM(total_revenue) FROM {GOLD_SCHEMA}.gold_monthly_sales), 0),
        2) AS silver_vs_gold_revenue_diff
""").display()

In [ ]:
# fk and scd2 integrity dq checks
spark.sql(f"""
    SELECT
        (SELECT COUNT(*) FROM {SILVER_SCHEMA}.fct_sales WHERE store_id IS NOT NULL
            AND store_id NOT IN (SELECT store_id FROM {SILVER_SCHEMA}.dim_stores)) AS orphan_store_fks,
        (SELECT COUNT(*) FROM {SILVER_SCHEMA}.fct_sales WHERE product_id IS NOT NULL
            AND product_id NOT IN (SELECT product_id FROM {SILVER_SCHEMA}.dim_products)) AS orphan_product_fks,
        (SELECT COUNT(*) FROM {SILVER_SCHEMA}.fct_sales WHERE vendor_id IS NOT NULL
            AND vendor_id NOT IN (SELECT vendor_id FROM {SILVER_SCHEMA}.dim_vendors)) AS orphan_vendor_fks,
        (SELECT COUNT(*) FROM {SILVER_SCHEMA}.dim_stores WHERE valid_from >= valid_to) AS stores_bad_date_range,
        (SELECT COUNT(*) FROM {SILVER_SCHEMA}.dim_products WHERE valid_from >= valid_to) AS products_bad_date_range,
        (SELECT COUNT(*) FROM {SILVER_SCHEMA}.dim_vendors WHERE valid_from >= valid_to) AS vendors_bad_date_range
""").display()

In [ ]:
# business rules check, all good if everything is zero
spark.sql(f"""
    SELECT
        (SELECT COUNT(*) FROM {SILVER_SCHEMA}.fct_sales
            WHERE sale_date > current_date()) AS future_dated_sales,
        (SELECT COUNT(*) FROM {SILVER_SCHEMA}.fct_sales
            WHERE profit IS NOT NULL AND state_bottle_cost IS NOT NULL
            AND ABS(profit - (sale_dollars - state_bottle_cost * bottles_sold)) > 0.01) AS bad_profit_calc,
        (SELECT COUNT(*) FROM (
            SELECT invoice_line_no FROM {SILVER_SCHEMA}.fct_sales
            GROUP BY invoice_line_no HAVING COUNT(*) > 1)) AS duplicate_invoices
""").display()